# Coletor de Dados: Hand Landmarks para CSV

Este notebook permite capturar as coordenadas dos marcos das mãos (hand landmarks) usando a webcam e salvá-los em um arquivo CSV. Esses dados podem ser usados posteriormente para treinar um modelo de classificação de gestos.

### Instruções:
1. Defina a `TARGET_LABEL` (etiqueta) para o gesto que você vai fazer.
2. Execute o loop da webcam.
3. Pressione **'s'** para salvar os marcos atuais da mão no CSV.
4. Pressione **'q'** para sair.

In [8]:
import cv2
import mediapipe as mp
import numpy as np
import csv
import os
import time

print("Bibliotecas carregadas!")


Bibliotecas carregadas!


### Configurações de Coleta
Altere o `TARGET_LABEL` conforme o gesto que estiver gravando.

In [9]:
TARGET_LABEL = "maoaberta"
CSV_FILE = "hand_landmarks_data.csv"
MODEL_PATH = "hand_landmarker.task"

print(f"Coletando dados para a label: {TARGET_LABEL}")
print(f"Arquivo de destino: {CSV_FILE}")


Coletando dados para a label: maoaberta
Arquivo de destino: hand_landmarks_data.csv


### Funções Auxiliares

In [10]:
def draw_hand_landmarks(frame, hand_landmarks):
    """Desenha os pontos e conexões da mão manualmente usando OpenCV."""
    HAND_CONNECTIONS = [
        (0, 1), (1, 2), (2, 3), (3, 4),
        (0, 5), (5, 6), (6, 7), (7, 8),
        (5, 9), (9, 10), (10, 11), (11, 12),
        (9, 13), (13, 14), (14, 15), (15, 16),
        (13, 17), (17, 18), (18, 19), (19, 20), (0, 17)
    ]

    h, w, _ = frame.shape
    for connection in HAND_CONNECTIONS:
        start_idx, end_idx = connection
        start_point = (int(hand_landmarks[start_idx].x * w), int(hand_landmarks[start_idx].y * h))
        end_point = (int(hand_landmarks[end_idx].x * w), int(hand_landmarks[end_idx].y * h))
        cv2.line(frame, start_point, end_point, (255, 255, 255), 2)

    for landmark in hand_landmarks:
        cx, cy = int(landmark.x * w), int(landmark.y * h)
        cv2.circle(frame, (cx, cy), 5, (0, 0, 255), -1)

def save_landmarks_to_csv(landmarks, label, filename):
    """Salva os marcos no arquivo CSV."""
    file_exists = os.path.isfile(filename)
    with open(filename, mode='a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:

            header = ['label']
            for i in range(21):
                header.extend([f'x{i}', f'y{i}', f'z{i}'])
            writer.writerow(header)

        row = [label]
        for lm in landmarks:
            row.extend([lm.x, lm.y, lm.z])
        writer.writerow(row)


### Loop de Captura e Gravação

- **'s'**: Salva os dados atuais de TODAS as mãos detectadas no CSV.
- **'q'**: Sai do loop.

In [11]:
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=1
)

cap = cv2.VideoCapture(0)
last_timestamp_ms = 0
count_saved = 0

print("Iniciando webcam... Pressione 's' para salvar e 'q' para sair.")

try:
    with HandLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                print("Ignorando frame vazio da câmera.")
                continue

            frame = cv2.flip(frame, 1)
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

            timestamp_ms = int(time.time() * 1000)
            if timestamp_ms <= last_timestamp_ms:
                timestamp_ms = last_timestamp_ms + 1
            last_timestamp_ms = timestamp_ms

            all_current_landmarks = []
            try:
                result = landmarker.detect_for_video(mp_image, timestamp_ms)
                if result.hand_landmarks:
                    all_current_landmarks = result.hand_landmarks
                    for hand_landmarks in all_current_landmarks:
                        draw_hand_landmarks(frame, hand_landmarks)

                    cv2.putText(frame, f"{len(all_current_landmarks)} Mao(s) - Label: {TARGET_LABEL}", (10, 30),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                else:
                    cv2.putText(frame, "Mao nao detectada", (10, 30),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            except Exception as e:
                print(f"Erro no MediaPipe: {e}")
                continue

            cv2.putText(frame, f"Total Salvos: {count_saved}", (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

            cv2.imshow('Coletor de Hand Landmarks (Multi-Hand)', frame)

            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            elif key == ord('s'):
                if all_current_landmarks:
                    for hand_lms in all_current_landmarks:
                        save_landmarks_to_csv(hand_lms, TARGET_LABEL, CSV_FILE)
                        count_saved += 1
                    print(f"Salvo {len(all_current_landmarks)} mao(s). Total acumulado: {count_saved}")
                else:
                    print("Nenhuma mao detectada no frame para salvar!")
finally:
    cap.release()
    cv2.destroyAllWindows()

    cv2.waitKey(1)
    print(f"Processo finalizado. Amostras totais nesta sessao: {count_saved}")


Iniciando webcam... Pressione 's' para salvar e 'q' para sair.
Processo finalizado. Amostras totais nesta sessao: 0
